In [ ]:
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Yêu cầu nhập trực tiếp từ bàn phím ngay trong Jupyter Notebook
TARGET_AGE = input(">> Nhập độ tuổi cần tra cứu (VD: 30): ").strip()
TARGET_ZIP = input(">> Nhập mã bưu điện cần tra cứu (VD: 70100): ").strip()
API_URL = "http://127.0.0.1:8001/anonymize"

print(f"\n[-] Đang truy vấn mạng phân tán P2P: Tuổi {TARGET_AGE}, Mã vùng {TARGET_ZIP}...")
response = requests.get(API_URL, params={"age": TARGET_AGE, "zip_code": TARGET_ZIP})

if response.status_code == 200:
    data = response.json()
    print("✅ Truy vấn thành công!")
    # Lấy dữ liệu ĐÃ LÀM MỜ từ API (Không phải dữ liệu nhập vào)
    gen_age = data['generalized_data'].get('age', data['generalized_data'].get('Age'))
    gen_zip = data['generalized_data'].get('zip_code', data['generalized_data'].get('ZipCode'))
    print(f"👉 Kết quả an toàn trả về: Tuổi {gen_age}, Mã vùng {gen_zip}")
else:
    print("❌ Lỗi kết nối đến Node 8001")

In [ ]:
import plotly.graph_objects as go

if response.status_code == 200:
    contributions = data.get("node_contributions", {})
    total_found = sum(contributions.values())
    
    if total_found > 0:
        labels = [f"Node {node}" for node, count in contributions.items() if count > 0]
        sizes = [count for count in contributions.values() if count > 0]
        
        # Tạo biểu đồ Donut Chart (Tròn rỗng ruột - nhìn hiện đại hơn)
        fig = go.Figure(data=[go.Pie(labels=labels, values=sizes, hole=.4, 
                                     hoverinfo="label+percent+value", 
                                     textinfo="label+percent")])
        
        fig.update_layout(title_text=f"<b>Tỷ lệ phân bổ dữ liệu trên mạng P2P</b><br>Tổng số người gom được: {data['total_count']}",
                          title_x=0.5, font=dict(size=14))
        fig.show()
    else:
        # Biểu đồ trạng thái trống (Empty State)
        labels = [f"Node {node}" for node in contributions.keys()]
        sizes = [0 for _ in labels]
        
        fig = go.Figure(data=[go.Bar(x=labels, y=sizes, marker_color='lightgrey')])
        fig.update_layout(title_text="<b>[KHÔNG TÌM THẤY DỮ LIỆU NÀO KHỚP]</b>", 
                          title_font_color="red", title_x=0.5,
                          yaxis=dict(range=[0, 5], title="Số lượng bản ghi"))
        fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

if response.status_code == 200:
    info_loss = data.get("info_loss", 0)
    iterations = data.get("iterations", 0)
    
    # Đóng gói dữ liệu vào DataFrame để Plotly xử lý
    df = pd.DataFrame({
        "Chỉ số": ["Mất mát thông tin (Info Loss)", "Số vòng lặp (Iterations)"],
        "Giá trị": [info_loss, iterations]
    })
    
    # Vẽ biểu đồ cột tương tác
    fig = px.bar(df, x="Chỉ số", y="Giá trị", color="Chỉ số", text="Giá trị",
                 color_discrete_sequence=["#FF6B6B", "#4ECDC4"])
    
    # Tùy chỉnh giao diện
    fig.update_traces(texttemplate='<b>%{text}</b>', textposition='outside', 
                      hovertemplate="%{x}: %{y}<extra></extra>")
    fig.update_layout(title_text="<b>Phân tích chi phí hiệu năng & Độ móp méo dữ liệu</b>",
                      title_x=0.5, showlegend=False, 
                      yaxis=dict(range=[0, max(info_loss, iterations) + 2], title="Số lượng"))
    fig.show()